In [ ]:
import pandas as pd

df=pd.read_csv('/Users/saqeeahmed/Desktop/Application_Build_25/netflix_data_analysis/netflix_titles.csv')


In [ ]:
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
df.isnull().sum()

In [ ]:
df.describe(include='all')

In [ ]:
df[(df['country']=='United States')].groupby('type').count()

In [ ]:
df['country'].value_counts().head(10)

In [ ]:
# India-র মোট মুভির সংখ্যা
egypt_movies = df[(df['country'] == 'Egypt') & (df['type'] == 'Movie')]
print("মোট সংখ্যা:", len(egypt_movies))

# ১ম ৫টির নাম
egypt_movies['title'].head(5)

In [ ]:
len(df[df['director'] == 'Marcus Raboy'])

In [ ]:
srk_movies = df[df['cast'].str.contains('Shah Rukh Khan', na=False)]
srk_movies[['title', 'release_year']]


In [125]:
df.groupby('type').count()

,show_id,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,rating_int
type,,,,,,,,,,,,
Movie,6131,6131,5943,5656,5691,6131,6131,6129,6131,6131,6131,6129
TV Show,2676,2676,230,2326,2285,2666,2676,2674,2676,2676,2676,2674


In [126]:
df[df['listed_in'].str.contains('Documentaries', na=False)]['title'].count()

np.int64(869)

In [129]:
df[(df['type']=='Movie') & (df['release_year'].between(2020,2024))]['title'].count()

np.int64(794)

In [131]:
# ১. শুধু Movie ফিল্টার করা
movies = df[df['type'] == 'Movie'].copy()

# ২. 'min' লেখাটি সরাতে হবে এবং Integer-এ নিতে হবে
movies['duration_int'] = movies['duration'].astype(str).str.replace(' min', '').astype(float)

# ৩. সর্বোচ্চ ডিউরেশনের মুভি বের করা
longest = movies[movies['duration_int'] == movies['duration_int'].max()]
longest[['title', 'duration_int']]

,title,duration_int
4253,Black Mirror: Bandersnatch,312.0


In [132]:
# NaN বাদ দিয়ে সব পরিচালক গণনা
df['director'].dropna().value_counts().head(1)

director
Rajiv Chilaka    19
Name: count, dtype: int64

In [133]:
# date_added-কে datetime ফরম্যাটে রূপান্তর করা
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())

# মাস বের করে গণনা করা
df['date_added'].dt.month_name().value_counts()

date_added
July         827
December     813
September    770
April        764
October      760
August       755
March        742
January      738
June         728
November     705
May          632
February     563
Name: count, dtype: int64

In [ ]:
print('oldest movie release year' ,df['release_year'].min())
print('latest movie release year' ,df['release_year'].max())
print('average movie release year' ,df['release_year'].mean())


In [ ]:
df[(df['cast']=='David Attenborough')]['title']

In [ ]:
df[df['director']=='Rajiv Chilaka']['title'].count()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df[(df['country']=='United States')].count()['type']

In [141]:
import json
import pandas as pd
import numpy as np

# ১. Netflix Data Extraction
movies_df = df[df['type'] == 'Movie'].copy()
movies_df['duration_num'] = pd.to_numeric(movies_df['duration'].astype(str).str.replace(' min', '', regex=False), errors='coerce')
avg_dur = round(float(movies_df['duration_num'].mean()), 1) if not movies_df['duration_num'].empty else 0.0

def clean_dict(d):
    return {str(k): int(v) for k, v in d.items() if pd.notna(k) and str(k).strip() != ''}

type_counts = clean_dict(df['type'].value_counts().to_dict())
top_countries = clean_dict(df['country'].dropna().value_counts().head(10).to_dict())
top_genres = clean_dict(df['listed_in'].astype(str).str.split(', ').explode().value_counts().head(8).to_dict())
top_directors = clean_dict(df['director'].dropna().value_counts().head(8).to_dict())

david_titles = df[df['cast'].astype(str).str.contains('David Attenborough', na=False)]['title'].tolist()

netflix_data = {
    "total_titles": int(len(df)),
    "total_movies": int((df['type'] == 'Movie').sum()),
    "total_tv": int((df['type'] == 'TV Show').sum()),
    "avg_movie_duration": avg_dur,
    "oldest_year": int(df['release_year'].min()) if 'release_year' in df else 0,
    "latest_year": int(df['release_year'].max()) if 'release_year' in df else 0,
    "rajiv_chilaka_count": int((df['director'] == 'Rajiv Chilaka').sum()),
    "marcus_raboy_count": int((df['director'] == 'Marcus Raboy').sum()),
    "srk_count": int(df['cast'].astype(str).str.contains('Shah Rukh Khan', na=False).sum()),
    "david_attenborough_titles": david_titles,
    "type_counts": type_counts,
    "top_countries": top_countries,
    "top_genres": top_genres,
    "top_directors": top_directors
}

json_payload = json.dumps(netflix_data)

# ২. HTML Dashboard Structure
html_template = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Netflix Content Analysis | Portfolio</title>
<link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;600;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<style>
  :root {
    --bg: #0b0f17;
    --card-bg: #141c2b;
    --card-border: #1e293b;
    --text-main: #f8fafc;
    --text-muted: #94a3b8;
    --accent-red: #e50914;
    --accent-cyan: #38bdf8;
    --accent-purple: #c084fc;
    --font-sans: 'Plus Jakarta Sans', sans-serif;
    --font-mono: 'JetBrains Mono', monospace;
  }

  * { box-sizing: border-box; margin: 0; padding: 0; }
  html { scroll-behavior: smooth; }
  body {
    background-color: var(--bg);
    color: var(--text-main);
    font-family: var(--font-sans);
    line-height: 1.6;
    padding-bottom: 60px;
  }

  /* Sticky Navigation Bar */
  nav.navbar {
    position: sticky; top: 0; z-index: 1000;
    background: rgba(11, 15, 23, 0.85);
    backdrop-filter: blur(12px);
    border-bottom: 1px solid var(--card-border);
    padding: 14px 0;
  }
  .nav-container { max-width: 1150px; margin: 0 auto; padding: 0 24px; display: flex; justify-content: space-between; align-items: center; }
  .nav-logo { font-size: 16px; font-weight: 800; color: #fff; text-decoration: none; display: flex; align-items: center; gap: 8px; }
  .nav-logo span { color: var(--accent-red); }
  .nav-links { display: flex; gap: 20px; list-style: none; }
  .nav-links a { color: var(--text-muted); text-decoration: none; font-size: 13px; font-family: var(--font-mono); transition: color 0.2s; }
  .nav-links a:hover { color: var(--accent-red); }

  /* Professional Hero Section */
  header.hero {
    padding: 70px 0 50px;
    border-bottom: 1px solid var(--card-border);
    background: radial-gradient(circle at top right, rgba(229, 9, 20, 0.15), transparent 60%);
  }
  .container { max-width: 1150px; margin: 0 auto; padding: 0 24px; }
  
  .badge {
    display: inline-flex; align-items: center; gap: 8px;
    background: rgba(229, 9, 20, 0.1); border: 1px solid var(--accent-red);
    color: var(--accent-red); padding: 6px 14px; border-radius: 99px;
    font-size: 12px; font-family: var(--font-mono); text-transform: uppercase;
  }
  .badge .dot { width: 6px; height: 6px; background: var(--accent-red); border-radius: 50%; box-shadow: 0 0 8px var(--accent-red); }

  h1 { font-size: 42px; font-weight: 800; margin: 18px 0 10px; letter-spacing: -0.02em; }
  p.subtitle { color: var(--text-muted); font-size: 16px; max-width: 700px; margin-bottom: 24px; }

  .hero-stats { display: flex; gap: 24px; flex-wrap: wrap; margin-top: 10px; }
  .stat-pill { background: var(--card-bg); border: 1px solid var(--card-border); padding: 8px 16px; border-radius: 8px; font-size: 13px; font-family: var(--font-mono); color: #fff; }
  .stat-pill span { color: var(--accent-red); font-weight: 700; }

  .section-title { font-size: 20px; font-weight: 700; margin: 50px 0 20px; display: flex; align-items: center; gap: 10px; scroll-margin-top: 80px; }
  .section-title::before { content: ''; width: 4px; height: 20px; background: var(--accent-red); border-radius: 2px; }

  .kpi-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 16px; }
  .kpi-card { background: var(--card-bg); border: 1px solid var(--card-border); padding: 22px; border-radius: 14px; }
  .kpi-val { font-size: 28px; font-weight: 800; color: #fff; }
  .kpi-val.highlight { color: var(--accent-red); }
  .kpi-lbl { font-size: 12px; color: var(--text-muted); margin-top: 4px; font-family: var(--font-mono); text-transform: uppercase; }

  .chart-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(480px, 1fr)); gap: 20px; margin-top: 16px; }
  .chart-card { background: var(--card-bg); border: 1px solid var(--card-border); padding: 22px; border-radius: 16px; }
  .chart-card h3 { font-size: 15px; font-weight: 600; margin-bottom: 4px; color: var(--text-main); }
  .chart-card p { font-size: 12px; color: var(--text-muted); margin-bottom: 16px; }
  .chart-container { position: relative; height: 260px; width: 100%; }

  .query-box { background: var(--card-bg); border: 1px solid var(--card-border); padding: 24px; border-radius: 16px; margin-top: 20px; }
  .query-box h3 { font-size: 15px; font-weight: 600; color: var(--accent-cyan); margin-bottom: 12px; }
  .query-box ul { list-style: none; display: grid; grid-template-columns: repeat(auto-fill, minmax(280px, 1fr)); gap: 10px; }
  .query-box li {
    background: rgba(255,255,255,0.03); border: 1px solid var(--card-border);
    padding: 10px 14px; border-radius: 8px; font-size: 13.5px;
  }

  footer { text-align: center; margin-top: 60px; color: var(--text-muted); font-size: 13px; font-family: var(--font-mono); }

  @media (max-width: 768px) { .chart-grid { grid-template-columns: 1fr; } .nav-links { display: none; } }
</style>
</head>
<body>

<nav class="navbar">
  <div class="nav-container">
    <a href="#" class="nav-logo"><span>NETFLIX</span> CATALOG EDA</a>
    <ul class="nav-links">
      <li><a href="#overview">Overview</a></li>
      <li><a href="#queries">Query Highlights</a></li>
      <li><a href="#visuals">Visualizations</a></li>
      <li><a href="#featured">Featured Cast</a></li>
    </ul>
  </div>
</nav>

<header class="hero">
  <div class="container">
    <div class="badge"><span class="dot"></span> Data Analysis Case File</div>
    <h1>Netflix Content Catalog Trends</h1>
    <p class="subtitle">An exploratory data analysis visualizing global streaming trends, format distributions, genres, and director portfolios from 8,800+ titles.</p>
    
    <div class="hero-stats">
      <div class="stat-pill">Source: <span>Netflix Titles Dataset</span></div>
      <div class="stat-pill">Analysis: <span>Pandas EDA</span></div>
      <div class="stat-pill">Records: <span>8,807 Movies & TV Shows</span></div>
    </div>
  </div>
</header>

<main class="container">

  <div class="section-title" id="overview">Overview Metrics</div>
  <div class="kpi-grid">
    <div class="kpi-card"><div class="kpi-val highlight" id="kpi-total">0</div><div class="kpi-lbl">Total Titles</div></div>
    <div class="kpi-card"><div class="kpi-val" id="kpi-movies">0</div><div class="kpi-lbl">Movies Count</div></div>
    <div class="kpi-card"><div class="kpi-val" id="kpi-tv">0</div><div class="kpi-lbl">TV Shows Count</div></div>
    <div class="kpi-card"><div class="kpi-val" id="kpi-runtime">0</div><div class="kpi-lbl">Avg. Movie Runtime</div></div>
  </div>

  <div class="section-title" id="queries">Notebook Query Highlights</div>
  <div class="kpi-grid">
    <div class="kpi-card"><div class="kpi-val" id="q-oldest">0</div><div class="kpi-lbl">Oldest Release Year</div></div>
    <div class="kpi-card"><div class="kpi-val" id="q-latest">0</div><div class="kpi-lbl">Latest Release Year</div></div>
    <div class="kpi-card"><div class="kpi-val" id="q-rajiv">0</div><div class="kpi-lbl">Rajiv Chilaka Titles</div></div>
    <div class="kpi-card"><div class="kpi-val" id="q-marcus">0</div><div class="kpi-lbl">Marcus Raboy Titles</div></div>
  </div>

  <div class="section-title" id="visuals">Visual Explorations</div>
  <div class="chart-grid">
    <div class="chart-card">
      <h3>Movie vs TV Show Ratio</h3>
      <p>Breakdown of catalog content format ratio.</p>
      <div class="chart-container"><canvas id="typeChart"></canvas></div>
    </div>

    <div class="chart-card">
      <h3>Top 10 Producing Countries</h3>
      <p>Dominant national origins across titles.</p>
      <div class="chart-container"><canvas id="countryChart"></canvas></div>
    </div>

    <div class="chart-card">
      <h3>Top Categories & Genres</h3>
      <p>Most frequently occurring genre tags.</p>
      <div class="chart-container"><canvas id="genreChart"></canvas></div>
    </div>

    <div class="chart-card">
      <h3>Most Prolific Directors</h3>
      <p>Top directors with highest content count.</p>
      <div class="chart-container"><canvas id="directorChart"></canvas></div>
    </div>
  </div>

  <div class="query-box" id="featured">
    <h3>🎬 Titles Starring David Attenborough (Notebook Query Result)</h3>
    <ul id="david-list"></ul>
  </div>

</main>

<footer>Dataset Analysis generated directly via Python Pandas Notebook</footer>

<script>
window.addEventListener('DOMContentLoaded', () => {
  try {
    const DATA = __JSON_PAYLOAD__;

    document.getElementById('kpi-total').innerText = DATA.total_titles.toLocaleString();
    document.getElementById('kpi-movies').innerText = DATA.total_movies.toLocaleString();
    document.getElementById('kpi-tv').innerText = DATA.total_tv.toLocaleString();
    document.getElementById('kpi-runtime').innerText = DATA.avg_movie_duration + ' min';

    document.getElementById('q-oldest').innerText = DATA.oldest_year;
    document.getElementById('q-latest').innerText = DATA.latest_year;
    document.getElementById('q-rajiv').innerText = DATA.rajiv_chilaka_count;
    document.getElementById('q-marcus').innerText = DATA.marcus_raboy_count;

    const davidUl = document.getElementById('david-list');
    DATA.david_attenborough_titles.forEach(t => {
      const li = document.createElement('li');
      li.innerText = '• ' + t;
      davidUl.appendChild(li);
    });

    Chart.defaults.color = '#94a3b8';
    Chart.defaults.font.family = "'Plus Jakarta Sans', sans-serif";

    // 1. Donut
    new Chart(document.getElementById('typeChart'), {
      type: 'doughnut',
      data: {
        labels: Object.keys(DATA.type_counts),
        datasets: [{ data: Object.values(DATA.type_counts), backgroundColor: ['#e50914', '#38bdf8'], borderWidth: 0 }]
      },
      options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { position: 'bottom' } } }
    });

    // 2. Countries
    new Chart(document.getElementById('countryChart'), {
      type: 'bar',
      data: {
        labels: Object.keys(DATA.top_countries),
        datasets: [{ data: Object.values(DATA.top_countries), backgroundColor: '#e50914', borderRadius: 4 }]
      },
      options: { responsive: true, maintainAspectRatio: false, indexAxis: 'y', plugins: { legend: { display: false } } }
    });

    // 3. Genres
    new Chart(document.getElementById('genreChart'), {
      type: 'bar',
      data: {
        labels: Object.keys(DATA.top_genres),
        datasets: [{ data: Object.values(DATA.top_genres), backgroundColor: '#c084fc', borderRadius: 4 }]
      },
      options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { display: false } } }
    });

    // 4. Directors
    new Chart(document.getElementById('directorChart'), {
      type: 'bar',
      data: {
        labels: Object.keys(DATA.top_directors),
        datasets: [{ data: Object.values(DATA.top_directors), backgroundColor: '#38bdf8', borderRadius: 4 }]
      },
      options: { responsive: true, maintainAspectRatio: false, indexAxis: 'y', plugins: { legend: { display: false } } }
    });

  } catch (err) {
    console.error("Dashboard error:", err);
  }
});
</script>

</body>
</html>
"""

final_html = html_template.replace('__JSON_PAYLOAD__', json_payload)

with open("index.html", "w", encoding="utf-8") as f:
    f.write(final_html)

print("🚀 'index.html' (Netflix) with Sticky Navbar & Hero section created successfully!")

🚀 'index.html' (Netflix) with Sticky Navbar & Hero section created successfully!
